# DSAR × Spark Declarative Pipelines · 02 · Zero-downtime erasure

A DSAR erasure request has arrived. This notebook erases the subject from
**every layer** the SDP pipeline produced — `raw_user` (bronze source),
`bronze_user`, `silver_user`, `gold_user` — and **physically purges** the raw
bytes, **while the pipeline keeps running**.

No pipeline stop. No full refresh. That works only because `01_sdp_pipeline` set
`skipChangeCommits` on every streaming hop: each stream skips the non-append
commit this notebook creates, instead of failing.

### Match strategy (email-first, key-resolved)

The DSAR intake gives us an **email**. But bronze/silver/gold have already
**redacted the email**, so we cannot match on it downstream. We resolve the
email → the stable `user_id` from the **raw** layer once, then erase by
`user_id` at every layer. (`user_id` is a non-PII business key, so it survives
obfuscation — this is why the pipeline keeps it unmasked.)

### Two modes (same as the base erasure layer)

- **DELETE** — remove the subject's rows entirely (right-to-delete).
- **OBFUSCATE** — keep the row, redact PII cells (opt-out / do-not-sell).

## 0. Configuration

In [ ]:
dbutils.widgets.removeAll()
dbutils.widgets.text("catalog", "dkushari_uc", "1 Catalog")
dbutils.widgets.text("schema", "allegiant_air_sdp_dsar", "2 Schema (isolated SDP demo)")
dbutils.widgets.text("subject_email", "", "3 Subject email (from DSAR intake)")
dbutils.widgets.dropdown("request_type", "DELETE", ["DELETE", "OBFUSCATE"], "4 Request type")
dbutils.widgets.text("redaction_token", "***REDACTED***", "5 Redaction token")
dbutils.widgets.dropdown("dry_run", "true", ["true", "false"], "6 Dry run (preview only)")
dbutils.widgets.dropdown("do_purge", "true", ["true", "false"], "7 Physically VACUUM after erase")

CATALOG = dbutils.widgets.get("catalog").strip()
SCHEMA  = dbutils.widgets.get("schema").strip()
FQ      = f"{CATALOG}.{SCHEMA}"
EMAIL   = dbutils.widgets.get("subject_email").strip().lower()
RTYPE   = dbutils.widgets.get("request_type").strip().upper()
TOKEN   = dbutils.widgets.get("redaction_token")
DRY     = dbutils.widgets.get("dry_run") == "true"
PURGE   = dbutils.widgets.get("do_purge") == "true"

assert EMAIL, "Set subject_email (run 00 to get the demo subject's email)."
print(f"Subject email : {EMAIL}")
print(f"Request type  : {RTYPE}")
print(f"Dry run       : {DRY}  (no changes written)" if DRY else f"Dry run       : {DRY}  (LIVE — will write)")
print(f"Physical purge: {PURGE}")

## 1. Resolve email → user_id (from the raw layer)

Email is still present in `raw_user` (the only layer that hasn't been obfuscated).
We resolve it to the stable `user_id`, which is what every downstream layer keys
on. If the subject was already erased from raw, we fall back to any layer that
still has a mapping (defensive).

In [ ]:
from pyspark.sql import functions as F

def sqlstr(s):
    return "'" + str(s).replace("'", "''") + "'"

# raw_user still has cleartext email
ids = (spark.table(f"{FQ}.raw_user")
       .where(F.lower("email") == EMAIL)
       .select("user_id").distinct())
user_ids = [r["user_id"] for r in ids.collect()]

if not user_ids:
    raise ValueError(f"No user_id found for {EMAIL} in raw_user. "
                     "Either the email is wrong or the subject was already erased.")

USER_IDS = user_ids
in_list = ", ".join(sqlstr(u) for u in USER_IDS)
print("Resolved user_id(s):", USER_IDS)
print("Match predicate     : user_id IN (", in_list, ")")

## 2. What we will touch (pre-count every layer)

Every table keys on `user_id`, so the predicate is uniform.

**Base tables vs. the derived MV.** `raw_user`, `bronze_user`, `silver_user` are
Delta **tables** we erase directly. `gold_user` is a **materialized view** derived
from bronze — you cannot `DELETE` from a view (`EXPECT_TABLE_NOT_VIEW`), and you
should not: it is *recomputed* from the erased base tables. So we erase + purge the
three base tables, then **refresh** the gold MV so its stored result drops the
subject too.

In [ ]:
BASE_TABLES = ["raw_user", "bronze_user", "silver_user"]  # erase + purge these
MV = "gold_user"                                           # derived — refresh, don't delete
ALL_LAYERS = BASE_TABLES + [MV]
pred = f"user_id IN ({in_list})"

print("Rows matching the subject, per layer:")
counts = {}
for t in ALL_LAYERS:
    try:
        n = spark.sql(f"SELECT count(*) c FROM {FQ}.{t} WHERE {pred}").collect()[0]["c"]
    except Exception as e:
        n = f"(missing: {str(e).splitlines()[0][:60]})"
    counts[t] = n
    kind = "(MV, derived)" if t == MV else "(base table)"
    print(f"  {t:<14} {n}   {kind}")

## 3. Erase the base tables

**Top-down (silver → bronze → raw)** so no layer momentarily references a subject
that a downstream layer already dropped. We do **not** touch `gold_user` here — it
is a materialized view and is refreshed in section 5.

- **DELETE** → `DELETE FROM <base table> WHERE user_id IN (...)`.
- **OBFUSCATE** → keep rows, redact the PII columns. Only `raw_user` still holds
  cleartext PII (email, full_name, `profile_json`); the bronze mask in `01`
  already redacted those columns downstream. We record which tables were actually
  modified so the purge step (section 4) only compacts those.

`ERASE_TOKEN` is pinned to bronze's redaction token so an obfuscated `raw_user`
cell matches the token downstream already carries — the widget default and
bronze's constant are both `***REDACTED***`; if you change one, change both.
(For DELETE the token is irrelevant.)

Each DELETE/UPDATE is a **non-append commit** on a streaming source — exactly what
`skipChangeCommits` lets the two streaming hops shrug off.

In [ ]:
# Pin the erasure token to what bronze (notebook 01) writes, so an obfuscated
# raw_user cell is identical to the downstream redaction. Keep these in sync.
BRONZE_TOKEN = "***REDACTED***"
ERASE_TOKEN = TOKEN
if RTYPE == "OBFUSCATE" and TOKEN != BRONZE_TOKEN:
    print(f"WARNING: redaction_token '{TOKEN}' != bronze token '{BRONZE_TOKEN}'.")
    print("         raw_user would be redacted with a different token than downstream.")
    print(f"         Pinning to bronze token '{BRONZE_TOKEN}' for cross-layer consistency.")
    ERASE_TOKEN = BRONZE_TOKEN

def _mask_json(col):
    e = f"regexp_replace({col}, '(\"email\" *: *)\"[^\"]*\"', '$1\"{ERASE_TOKEN}\"')"
    e = f"regexp_replace({e}, '(\"name\" *: *)\"[^\"]*\"', '$1\"{ERASE_TOKEN}\"')"
    return e

# erase base tables only, top-down: silver -> bronze -> raw (gold MV handled in section 5)
ORDER = ["silver_user", "bronze_user", "raw_user"]
modified = []   # base tables we actually wrote to -> only these get purged

def erase_sql(table):
    if RTYPE == "DELETE":
        return f"DELETE FROM {FQ}.{table} WHERE {pred}"
    # OBFUSCATE — only raw_user still holds cleartext PII
    if table == "raw_user":
        sets = [
            f"email = {sqlstr(ERASE_TOKEN)}",
            f"full_name = {sqlstr(ERASE_TOKEN)}",
            f"profile_json = {_mask_json('profile_json')}",
        ]
        return f"UPDATE {FQ}.{table} SET {', '.join(sets)} WHERE {pred}"
    return None  # downstream already redacted; nothing to obfuscate

for t in ORDER:
    stmt = erase_sql(t)
    if stmt is None:
        print(f"[skip]  {t}: already redacted downstream (OBFUSCATE mode)")
        continue
    print(f"\n[{'DRY-RUN' if DRY else 'RUN'}] {stmt}")
    if not DRY:
        spark.sql(stmt)
        print("   done.")
    modified.append(t)

print("\nBase tables modified this run:", modified or "(none — dry run shows intent above)")

## 4. Physical purge of the base tables (CCPA "no trace")

`DELETE`/`UPDATE` only tombstone the old files. To make the bytes unrecoverable
we compact + VACUUM **the base tables we actually modified** (in DELETE mode all
three of raw/bronze/silver; in OBFUSCATE mode only `raw_user`, so we don't waste
I/O compacting untouched streaming tables). The gold MV is handled in section 5.

**Serverless gotcha** (same as the base `03_physical_purge`):
`spark.databricks.delta.retentionDurationCheck.enabled` is **not settable** on
serverless / Spark Connect. So we set the **table property**
`delta.deletedFileRetentionDuration = 'interval 0 hours'` and run a **plain
`VACUUM`** (no `RETAIN` clause). `VACUUM ... RETAIN 0 HOURS` still trips the
safety check; the property route works everywhere.

> This runs while the pipeline streams. VACUUMing a streaming source is safe here
> because `skipChangeCommits` streams don't need the tombstoned files.

In [ ]:
def purge(table):
    fqt = f"{FQ}.{table}"
    spark.sql(f"ALTER TABLE {fqt} SET TBLPROPERTIES ('delta.deletedFileRetentionDuration' = 'interval 0 hours')")
    try:
        spark.sql(f"REORG TABLE {fqt} APPLY (PURGE)")
    except Exception as e:
        print(f"   REORG skipped on {table}:", str(e).splitlines()[0][:80])
    spark.sql(f"VACUUM {fqt}")   # plain VACUUM — honours the table property
    print(f"   purged {table}")

if DRY:
    print("Dry run — skipping physical purge.")
elif not PURGE:
    print("do_purge=false — skipping VACUUM (soft-deleted only).")
elif not modified:
    print("No layers modified — nothing to purge.")
else:
    for t in modified:   # only compact what we actually wrote to
        try:
            purge(t)
        except Exception as e:
            print(f"   purge failed on {t}:", str(e).splitlines()[0][:80])

## 5. Refresh the gold materialized view

`gold_user` is a view derived from bronze, so we don't (and can't) `DELETE` from
it — but its **stored result** still holds the subject's aggregate row until it is
recomputed. For true CCPA "no trace" we refresh it so it re-derives from the now-
erased bronze.

The durable way to refresh an SDP-managed MV is to **trigger a pipeline update**
(the pipeline recomputes `gold_user` from bronze). We kick that off here via the
SDK. (If you prefer, you can instead run the pipeline from the UI — either way the
MV recomputes without the subject.)

> Why this is the correct model: an MV is *defined by its query over bronze*.
> Erase bronze, refresh the MV, and the subject is gone from gold by construction —
> no direct delete, and no streaming checkpoint state that could resurrect it.

In [ ]:
PIPELINE_ID = ""  # optional: set to your pipeline id to auto-refresh gold from here

if DRY:
    print("Dry run — gold MV not refreshed.")
elif not PIPELINE_ID:
    print("No PIPELINE_ID set. Refresh gold_user by running the pipeline (UI or SDK):")
    print("  - UI: open the pipeline and click 'Start' (or 'Full refresh' on gold_user)")
    print("  - SDK: w.pipelines.start_update(pipeline_id=..., full_refresh_selection=['gold_user'])")
    print("Until refreshed, gold_user still shows the subject's stored aggregate row.")
else:
    from databricks.sdk import WorkspaceClient
    w = WorkspaceClient()
    upd = w.pipelines.start_update(pipeline_id=PIPELINE_ID, full_refresh_selection=["gold_user"])
    print(f"Started pipeline update to refresh gold_user: {upd}")
    print("Wait for it to complete, then re-run section 6 to confirm gold is clean.")

## 6. Validate — no trace at any layer

After a live DELETE the base-table counts should be **0** immediately; `gold_user`
becomes 0 once the MV refresh (section 5) completes. (For OBFUSCATE, downstream
counts stay as-is but carry no PII; `raw_user` matches remain but are redacted — we
assert no cleartext email survives.)

In [ ]:
if DRY:
    print("Dry run — nothing changed; re-run with dry_run=false to erase.")
else:
    print("Post-erasure counts (user_id match):")
    for t in ALL_LAYERS:
        try:
            n = spark.sql(f"SELECT count(*) c FROM {FQ}.{t} WHERE {pred}").collect()[0]["c"]
            note = "  <- refresh gold MV (section 5) to zero this" if (t == MV and n) else ""
            print(f"  {t:<14} {n}{note}")
        except Exception as e:
            print(f"  {t:<14} (err) {str(e).splitlines()[0][:60]}")

    # cleartext email must be gone from raw regardless of mode
    left = spark.sql(f"SELECT count(*) c FROM {FQ}.raw_user WHERE lower(email) = {sqlstr(EMAIL)}").collect()[0]["c"]
    print(f"\nCleartext email still in raw_user: {left}  (must be 0)")
    assert left == 0, "Cleartext email survived — erasure incomplete!"

    # base tables must have no subject rows after a DELETE
    if RTYPE == "DELETE":
        for t in BASE_TABLES:
            n = spark.sql(f"SELECT count(*) c FROM {FQ}.{t} WHERE {pred}").collect()[0]["c"]
            assert n == 0, f"{t} still has {n} subject rows after DELETE!"
    print("PASS — subject erased from all base tables with no cleartext trace.")
    print("      (gold MV drops the subject on its next refresh — section 5.)")

## 7. Confirm the pipeline is still healthy

The whole point: the pipeline **did not fail**. After running this, open the
pipeline in the UI and confirm the latest update is still `RUNNING`/`COMPLETED`
with no `append-only source` error. New raw events keep flowing through
bronze → silver → gold; the erasure commit was skipped, not fatal.

You can also append a fresh raw event and watch it propagate to prove liveness:

In [ ]:
# Optional liveness probe: append one new raw event and confirm the pipeline
# processes it (run the pipeline / wait for the next micro-batch, then check gold).
from pyspark.sql import functions as F
probe = (spark.range(1)
         .select(
             F.lit("PROBE-0").alias("event_id"),
             F.lit("U999999").alias("user_id"),
             F.lit("probe@example.com").alias("email"),
             F.lit("Probe User").alias("full_name"),
             F.lit('{"contact":{"email":"probe@example.com","name":"Probe User"},"loyalty":{"tier":"gold","ltv":1.0}}').alias("profile_json"),
             F.lit(1.0).alias("revenue"),
             F.current_timestamp().alias("event_ts"),
             F.current_timestamp().alias("_ingest_ts")))
if not DRY:
    probe.write.mode("append").saveAsTable(f"{FQ}.raw_user")
    print("Appended probe event U999999. Run the pipeline; it should appear in gold_user,")
    print("proving the stream is alive after the erasure.")
else:
    print("Dry run — probe not appended.")

## 8. Idempotency — full refresh + repeated incrementals

Because erasure removes the subject from the **base tables** (raw/bronze/silver),
the pipeline is idempotent under any mix of incremental and full-refresh updates:

- **Incremental update** — bronze/silver read only new commits; the erased subject
  isn't in them, so it doesn't reappear. Re-running changes nothing.
- **Full refresh** — resets every table and re-reads `raw_user` from scratch. The
  subject is gone from raw, so it can't come back; gold recomputes clean.
- **Any sequence, any number of times** — incremental → full refresh → incremental
  → … all converge to the same state (subject absent, everyone else intact).

This was verified live: counts stay identical across repeated incrementals and a
full refresh, and the erased `user_id` never reappears. The one nuance: after an
erasure, run a pipeline update (incremental is enough for base tables; a **full
refresh** guarantees the gold MV also drops the subject — see section 5). Both are
safe to repeat.

In [ ]:
# Idempotency self-check: assert the subject is absent everywhere and re-running
# this notebook (or the pipeline) does not change counts.
if not DRY:
    print("Idempotency check — subject rows per layer (all must be 0):")
    ok = True
    for t in ALL_LAYERS:
        n = spark.sql(f"SELECT count(*) c FROM {FQ}.{t} WHERE {pred}").collect()[0]["c"]
        flag = "" if n == 0 else "  <- refresh gold MV (section 5) if this is gold_user"
        print(f"  {t:<14} {n}{flag}")
        if n and t in BASE_TABLES:
            ok = False
    print("Base tables idempotent & clean." if ok else "Base tables still have subject rows — re-run erase.")
else:
    print("Dry run — idempotency check skipped.")